# EcoPredict v3 - Notebook clean multi-departements

Ce notebook entraine un modele multi-sorties (conso, CO2) a partir des CSV DPE,
puis exporte un pipeline joblib reutilise par le backend FastAPI.

## Plan
1. Charger et fusionner les CSV
2. Nettoyer et preparer les features
3. Entrainement du pipeline
4. Evaluation rapide
5. Export joblib
6. Fonction de prediction locale

In [27]:
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)

RANDOM_STATE = 42
print('Imports OK')

Imports OK


In [28]:
DATA_FILES = [
    'dpe_logement_13.csv', 'dpe_logement_33.csv', 'dpe_logement_44.csv',
    'dpe_logement_59.csv', 'dpe_logement_63.csv', 'dpe_logement_67.csv',
    'dpe_logement_74.csv', 'dpe_logement_75.csv',
]

DEPT_TO_ZONE = {
    '59': 'H1', '67': 'H1', '74': 'H1',
    '44': 'H2', '33': 'H2', '75': 'H2', '63': 'H2',
    '13': 'H3',
}

def read_dpe_csv(path):
    # 1) tentative rapide: parser C
    for enc in ('utf-8', 'latin1', 'cp1252'):
        try:
            return pd.read_csv(
                path,
                sep=';',
                encoding=enc,
                low_memory=False,
                on_bad_lines='skip',
            )
        except Exception:
            pass

    # 2) fallback robuste mais plus lent: parser Python
    for enc in ('utf-8', 'latin1', 'cp1252'):
        try:
            return pd.read_csv(
                path,
                sep=';',
                encoding=enc,
                on_bad_lines='skip',
                engine='python',
            )
        except Exception as exc:
            last_exc = exc

    raise last_exc

missing = [f for f in DATA_FILES if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f'CSV manquants: {missing}')

frames = []
for i, f in enumerate(DATA_FILES, start=1):
    dept = f.split('_')[-1].replace('.csv', '')
    print(f'[{i}/{len(DATA_FILES)}] lecture {f} ...')
    tmp = read_dpe_csv(f)
    tmp['code_departement'] = dept
    tmp['zone_climatique'] = DEPT_TO_ZONE.get(dept, 'H2')
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print('Rows:', len(df), 'Cols:', df.shape[1])

[1/8] lecture dpe_logement_13.csv ...
[2/8] lecture dpe_logement_33.csv ...
[3/8] lecture dpe_logement_44.csv ...
[4/8] lecture dpe_logement_59.csv ...
[5/8] lecture dpe_logement_63.csv ...
[6/8] lecture dpe_logement_67.csv ...
[7/8] lecture dpe_logement_74.csv ...
[8/8] lecture dpe_logement_75.csv ...
Rows: 3549130 Cols: 107


In [29]:
# Features ciblees (alignees avec engine.py)
FEATURES_NUM_RAW = [
    'surface_habitable_logement', 'annee_construction_dpe',
    'nombre_niveau_logement', 'nombre_niveau_immeuble', 'surface_habitable_immeuble'
]
FEATURES_NUM_ENVELOPE = [
    'surface_mur_totale', 'surface_mur_exterieur', 'surface_mur_deperditif',
    'u_mur_exterieur', 'surface_plancher_bas_totale', 'surface_plancher_bas_deperditif',
    'surface_plancher_haut_totale', 'surface_plancher_haut_deperditif',
    'u_baie_vitree', 'facteur_solaire_baie_vitree'
]
FEATURES_NUM_VITRAGE = [
    'surface_vitree_nord', 'surface_vitree_sud', 'surface_vitree_ouest', 'surface_vitree_est'
]
FEATURES_NUM_ALL = FEATURES_NUM_RAW + FEATURES_NUM_ENVELOPE + FEATURES_NUM_VITRAGE

FEATURES_ENGINEERED = [
    'is_rt2012', 'age_batiment', 'surface_vitree_totale',
    'ratio_vitrage', 'ratio_mur_deperditif', 'surface_par_niveau', 'compacite'
]

FEATURES_CAT_BASE = [
    'type_batiment_dpe', 'type_energie_chauffage', 'type_installation_chauffage',
    'type_generateur_chauffage', 'type_energie_ecs', 'type_generateur_ecs',
    'type_installation_ecs', 'type_isolation_mur_exterieur',
    'type_isolation_plancher_bas', 'type_isolation_plancher_haut',
    'materiaux_structure_mur_exterieur', 'type_vitrage', 'type_materiaux_menuiserie',
    'type_gaz_lame', 'type_ventilation', 'periode_construction_dpe',
    'chauffage_solaire', 'ecs_solaire', 'traversant', 'presence_balcon', 'classe_inertie'
]

FEATURES_CAT = FEATURES_CAT_BASE + ['zone_climatique', 'code_departement']
FEATURES_NUM = FEATURES_NUM_ALL + FEATURES_ENGINEERED
ALL_FEATURES = FEATURES_NUM + FEATURES_CAT

# Cibles canoniques utilisees ensuite par le pipeline
TARGETS = ['conso_5_usages_e_finale', 'emission_ges_5_usages']

# Alias observes dans les CSV ADEME
TARGET_ALIASES = {
    'conso_5_usages_e_finale': ['conso_5_usages_e_finale', 'conso_5_usages_ef_m2', 'conso_5_usages_ep_m2'],
    'emission_ges_5_usages': ['emission_ges_5_usages', 'emission_ges_5_usages_m2'],
}

In [30]:
def to_numeric_safe(frame, cols):
    out = frame.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def to_categorical_safe(frame, cols):
    out = frame.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = 'inconnu'
        out[c] = out[c].fillna('inconnu').astype(str)
    return out

# Harmonisation des cibles: choisir l'alias le plus renseigne
for canonical, aliases in TARGET_ALIASES.items():
    candidates = [col for col in aliases if col in df.columns]
    if not candidates:
        raise KeyError(f"Aucune colonne cible trouvee pour {canonical}. Alias testes: {aliases}")

    best_col = None
    best_score = -1
    for col in candidates:
        s = pd.to_numeric(df[col], errors='coerce')
        score = int(s.notna().sum())
        if score > best_score:
            best_col = col
            best_score = score

    df[canonical] = pd.to_numeric(df[best_col], errors='coerce')
    print(f"Target map: {canonical} <- {best_col} (valid={best_score})")

df = to_numeric_safe(df, FEATURES_NUM_ALL + TARGETS)
df = to_categorical_safe(df, FEATURES_CAT)

# Cibles valides
df = df.dropna(subset=TARGETS)
df = df[(df['conso_5_usages_e_finale'] > 0) & (df['emission_ges_5_usages'] >= 0)]

print('Rows apres nettoyage cibles:', len(df))
if len(df) == 0:
    raise ValueError('Aucune ligne exploitable apres nettoyage. Verifier mapping des cibles et qualite des CSV.')

Target map: conso_5_usages_e_finale <- conso_5_usages_ef_m2 (valid=2295435)
Target map: emission_ges_5_usages <- emission_ges_5_usages_m2 (valid=2295435)
Rows apres nettoyage cibles: 2295435


In [31]:
# Feature engineering (meme logique que engine.py)
df['is_rt2012'] = (df['annee_construction_dpe'] >= 2013).astype(int)
df['age_batiment'] = 2026 - df['annee_construction_dpe'].fillna(1990)
df['surface_vitree_totale'] = (
    df['surface_vitree_nord'].fillna(0) + df['surface_vitree_sud'].fillna(0) +
    df['surface_vitree_ouest'].fillna(0) + df['surface_vitree_est'].fillna(0)
)
df['ratio_vitrage'] = df['surface_vitree_totale'] / df['surface_habitable_logement'].clip(lower=1)
df['ratio_mur_deperditif'] = df['surface_mur_deperditif'] / df['surface_mur_totale'].clip(lower=1)
df['surface_par_niveau'] = df['surface_habitable_logement'] / df['nombre_niveau_logement'].clip(lower=1)
df['compacite'] = df['surface_mur_exterieur'] / df['surface_habitable_logement'].clip(lower=1)

# bornes simples pour reduire les outliers extremes
for c in ['ratio_vitrage', 'ratio_mur_deperditif', 'compacite']:
    df[c] = df[c].clip(lower=0, upper=df[c].quantile(0.99))

# Pour garder un notebook executable sur machine dev, on plafonne l'echantillon d'entrainement
MAX_ROWS_FOR_TRAIN = 400_000
if len(df) > MAX_ROWS_FOR_TRAIN:
    df = df.sample(MAX_ROWS_FOR_TRAIN, random_state=RANDOM_STATE)
    print(f'Sampling applique: {MAX_ROWS_FOR_TRAIN} lignes')

X = df[ALL_FEATURES].copy()
y = df[TARGETS].copy()
print('X:', X.shape, 'y:', y.shape)

Sampling applique: 400000 lignes
X: (400000, 49) y: (400000, 2)


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, FEATURES_NUM),
        ('cat', categorical_transformer, FEATURES_CAT),
    ]
)

base_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=1,
    tree_method='hist'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', MultiOutputRegressor(base_model))
])

print('Pipeline ready')

Pipeline ready


In [33]:
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

metrics = {
    'r2_conso': r2_score(y_test.iloc[:, 0], pred[:, 0]),
    'r2_co2': r2_score(y_test.iloc[:, 1], pred[:, 1]),
    'mae_conso': mean_absolute_error(y_test.iloc[:, 0], pred[:, 0]),
    'mae_co2': mean_absolute_error(y_test.iloc[:, 1], pred[:, 1]),
}

metrics

{'r2_conso': 0.792359512796931,
 'r2_co2': 0.829076221692846,
 'mae_conso': 23.50584219085391,
 'mae_co2': 4.82504252497041}

In [34]:
MODEL_PATH = 'ecopredict_v3_pipeline.joblib'
joblib.dump(pipeline, MODEL_PATH)
size_mb = os.path.getsize(MODEL_PATH) / 1024 / 1024
print(f'Pipeline exporte vers {MODEL_PATH} ({size_mb:.1f} MB)')

Pipeline exporte vers ecopredict_v3_pipeline.joblib (5.8 MB)


In [35]:
def classe_dpe(c):
    if c <= 50: return 'A'
    if c <= 90: return 'B'
    if c <= 150: return 'C'
    if c <= 230: return 'D'
    if c <= 330: return 'E'
    if c <= 450: return 'F'
    return 'G'

def predict_energy(input_dict, loaded_pipeline=None):
    p = loaded_pipeline if loaded_pipeline is not None else pipeline
    row = pd.DataFrame([input_dict])

    # securiser schema de colonnes
    for c in ALL_FEATURES:
        if c not in row.columns:
            row[c] = np.nan if c in FEATURES_NUM else 'inconnu'

    row = to_numeric_safe(row, FEATURES_NUM_ALL)
    row = to_categorical_safe(row, FEATURES_CAT)

    row['is_rt2012'] = (row['annee_construction_dpe'] >= 2013).astype(int)
    row['age_batiment'] = 2026 - row['annee_construction_dpe'].fillna(1990)
    row['surface_vitree_totale'] = row[['surface_vitree_nord', 'surface_vitree_sud', 'surface_vitree_ouest', 'surface_vitree_est']].fillna(0).sum(axis=1)
    row['ratio_vitrage'] = row['surface_vitree_totale'] / row['surface_habitable_logement'].clip(lower=1)
    row['ratio_mur_deperditif'] = row['surface_mur_deperditif'] / row['surface_mur_totale'].clip(lower=1)
    row['surface_par_niveau'] = row['surface_habitable_logement'] / row['nombre_niveau_logement'].clip(lower=1)
    row['compacite'] = row['surface_mur_exterieur'] / row['surface_habitable_logement'].clip(lower=1)

    pred = p.predict(row[ALL_FEATURES])[0]
    conso, co2 = float(pred[0]), float(pred[1])
    return {
        'conso_kwh_m2_an': round(conso, 1),
        'co2_kg_m2_an': round(co2, 1),
        'classe_dpe': classe_dpe(conso),
    }

print('predict_energy() ready')

predict_energy() ready


In [36]:
sample = {
    'surface_habitable_logement': 85,
    'annee_construction_dpe': 1975,
    'nombre_niveau_logement': 1,
    'nombre_niveau_immeuble': 4,
    'surface_habitable_immeuble': 340,
    'surface_mur_totale': 120,
    'surface_mur_exterieur': 90,
    'surface_mur_deperditif': 90,
    'u_mur_exterieur': 2.5,
    'surface_plancher_bas_totale': 85,
    'surface_plancher_bas_deperditif': 85,
    'surface_plancher_haut_totale': 0,
    'surface_plancher_haut_deperditif': 0,
    'u_baie_vitree': 3.3,
    'facteur_solaire_baie_vitree': 0.65,
    'surface_vitree_nord': 3,
    'surface_vitree_sud': 5,
    'surface_vitree_ouest': 2,
    'surface_vitree_est': 2,
    'type_batiment_dpe': 'appartement',
    'type_energie_chauffage': 'gaz',
    'type_installation_chauffage': 'collectif',
    'type_generateur_chauffage': 'chaudiere classique gaz',
    'type_energie_ecs': 'gaz',
    'type_generateur_ecs': 'chaudiere classique gaz',
    'type_installation_ecs': 'collectif',
    'type_isolation_mur_exterieur': 'non isole',
    'type_isolation_plancher_bas': 'non isole',
    'type_isolation_plancher_haut': 'inconnu',
    'materiaux_structure_mur_exterieur': 'beton',
    'type_vitrage': 'simple vitrage',
    'type_materiaux_menuiserie': 'bois',
    'type_gaz_lame': 'inconnu',
    'type_ventilation': 'Ventilation naturelle',
    'periode_construction_dpe': '1975-1977',
    'chauffage_solaire': '0',
    'ecs_solaire': '0',
    'traversant': 'inconnu',
    'presence_balcon': '1',
    'classe_inertie': 'lourde',
    'zone_climatique': 'H2',
    'code_departement': '75',
}

predict_energy(sample)

{'conso_kwh_m2_an': 374.1, 'co2_kg_m2_an': 72.7, 'classe_dpe': 'F'}

## Sorties principales
- un pipeline sklearn exporte dans `ecopredict_v3_pipeline.joblib`
- des metriques de performance (`r2`, `mae`)
- une fonction `predict_energy` pour tester des logements

Ce joblib est ensuite charge par `engine.py` dans l'application FastAPI.